In [ ]:
!pip install -q transformers datasets evaluate sentencepiece
!pip install -q torch
!pip install -q sentence-transformers
!pip install -q nltk spacy
!pip install bert_score
!pip install textstat
!python -m spacy download en_core_web_sm
!pip install -q scikit-learn scipy gensim networkx
!git clone https://github.com/neulab/BARTScore.git

In [ ]:
# ============================================================
# RALSS: Rhetorical-Aware Legal Summarization System
# Cleaned Research Implementation
# ============================================================

# ============================================================
# STANDARD LIBRARIES
# ============================================================

import os
import sys
import gc
import re
import json
import math

# ============================================================
# CORE LIBRARIES
# ============================================================

import numpy as np
from tqdm import tqdm

# ============================================================
# PYTORCH
# ============================================================

import torch
import torch.nn.functional as F

# ============================================================
# NLP LIBRARIES
# ============================================================

import nltk
import spacy
import evaluate

from nltk.corpus import stopwords

# ============================================================
# TRANSFORMERS
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)

from sentence_transformers import SentenceTransformer

# ============================================================
# SCIKIT-LEARN
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# BARTScore
# ============================================================

sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

from bart_score import BARTScorer

# ============================================================
# INITIAL SETUP
# ============================================================

nltk.download("punkt")
nltk.download("stopwords")

nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using Device: {device}")

# ============================================================
# LOAD DATASET
# ============================================================

DATASET_PATH = (
    "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"
)

with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

# ============================================================
# LOAD SUMMARIZATION MODEL
# ============================================================

MODEL_PATH = (
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH
).to(device)

summarizer_model.eval()

# ============================================================
# LOAD EVALUATION METRICS
# ============================================================

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# ============================================================
# LOAD BARTScore
# ============================================================

bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

# ============================================================
# LOAD FACTCC
# ============================================================

FACTCC_MODEL = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(
    FACTCC_MODEL
)

factcc_model = AutoModelForSequenceClassification.from_pretrained(
    FACTCC_MODEL
).to(device)

factcc_model.eval()

# ============================================================
# LOAD SBERT
# ============================================================

sbert_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device=device
)

 ============================================================
# LEGAL ROLE ORDER
# ============================================================

LEGAL_FLOW = [
    "PREAMBLE",
    "FAC",
    "ISSUE",
    "ARG_PETITIONER",
    "ARG_RESPONDENT",
    "RLC",
    "STA",
    "PRE_RELIED",
    "PRE_NOT_RELIED",
    "ANALYSIS",
    "RATIO",
    "RPC"
]

# ============================================================
# ROLE IMPORTANCE WEIGHTS
# ============================================================

ROLE_WEIGHTS = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PREAMBLE": 0.4,
    "NONE": 0.2
}

# ============================================================
# ABLATION CONFIGURATION
# ============================================================

ABLATION_VARIANTS = {

    "RALSS_FULL": {
        "use_position_weights": True,
        "use_dynamic_budget": True,
        "use_legal_order": True
    },

    "NO_POSITION": {
        "use_position_weights": False,
        "use_dynamic_budget": True,
        "use_legal_order": True
    },

    "NO_DYNAMIC": {
        "use_position_weights": True,
        "use_dynamic_budget": False,
        "use_legal_order": True
    },

    "NO_ORDER": {
        "use_position_weights": True,
        "use_dynamic_budget": True,
        "use_legal_order": False
    }
}

# ============================================================
# TEXT PREPROCESSING
# ============================================================

def preprocess(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ============================================================
# POSITIONAL WEIGHTING
# ============================================================

def position_weight(idx, total):

    if total <= 1:
        return 1.0

    relative = idx / total

    if relative < 0.15:
        return 1.2

    elif relative > 0.85:
        return 1.15

    return 1.0

# ============================================================
# SUMMARY LENGTH ESTIMATION
# ============================================================

def estimate_summary_length(
    sentences,
    ratio=0.25,
    min_len=8,
    max_len=30
):

    N = len(sentences)

    L = int(N * ratio)

    return max(
        min_len,
        min(L, max_len)
    )

# ============================================================
# HYBRID SCORING
# ============================================================

def compute_hybrid_scores(sentences):

    # --------------------------------------------------------
    # TF-IDF LEXICAL SCORING
    # --------------------------------------------------------

    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(
        sentences
    )

    tfidf_scores = np.sum(
        tfidf_matrix.toarray(),
        axis=1
    )

    # --------------------------------------------------------
    # SBERT SEMANTIC SCORING
    # --------------------------------------------------------

    sbert_embeddings = sbert_model.encode(
        sentences
    )

    document_embedding = np.mean(
        sbert_embeddings,
        axis=0
    )

    semantic_scores = cosine_similarity(
        sbert_embeddings,
        document_embedding.reshape(1, -1)
    ).flatten()

    # --------------------------------------------------------
    # NORMALIZATION
    # --------------------------------------------------------

    tfidf_scores = (
        tfidf_scores /
        (np.max(tfidf_scores) + 1e-8)
    )

    semantic_scores = (
        semantic_scores /
        (np.max(semantic_scores) + 1e-8)
    )

    # --------------------------------------------------------
    # HYBRID SCORE
    # --------------------------------------------------------

    hybrid_scores = (
        0.65 * tfidf_scores +
        0.35 * semantic_scores
    )

    return hybrid_scores

# ============================================================
# RALSS SENTENCE SELECTION
# ============================================================

def ralss_sentence_selection(
    sentences,
    roles,
    config,
    compression_ratio=0.25
):

    if len(sentences) == 0:
        return []

    # --------------------------------------------------------
    # SUMMARY LIMIT
    # --------------------------------------------------------

    summary_limit = estimate_summary_length(
        sentences,
        ratio=compression_ratio
    )

    # --------------------------------------------------------
    # BASE HYBRID SCORES
    # --------------------------------------------------------

    base_scores = compute_hybrid_scores(
        sentences
    )

    weighted_scores = []

    # --------------------------------------------------------
    # POSITION + LENGTH WEIGHTING
    # --------------------------------------------------------

    for i in range(len(sentences)):

        pos_weight = (
            position_weight(i, len(sentences))
            if config["use_position_weights"]
            else 1.0
        )

        length_bonus = (
            1 + (len(sentences[i].split()) / 100)
        )

        score = (
            base_scores[i] *
            pos_weight *
            length_bonus
        )

        weighted_scores.append(score)

    weighted_scores = np.array(weighted_scores)

    # --------------------------------------------------------
    # GROUP BY RHETORICAL ROLE
    # --------------------------------------------------------

    role_groups = {}

    for i, role in enumerate(roles):
        role_groups.setdefault(role, []).append(i)

    selected_indices = []

    # --------------------------------------------------------
    # DYNAMIC EVIDENCE BUDGET ALLOCATION
    # --------------------------------------------------------

    for role, idxs in role_groups.items():

        if config["use_dynamic_budget"]:

            role_weight = ROLE_WEIGHTS.get(
                role,
                0.2
            )

            role_budget = math.ceil(
                len(idxs) *
                role_weight *
                compression_ratio
            )

        else:

            role_budget = max(
                1,
                int(
                    len(sentences) *
                    compression_ratio /
                    len(role_groups)
                )
            )

        role_budget = max(role_budget, 1)

        ranked = sorted(
            idxs,
            key=lambda x: weighted_scores[x],
            reverse=True
        )

        selected_indices.extend(
            ranked[:role_budget]
        )

    # --------------------------------------------------------
    # FINAL SUMMARY LIMITING
    # --------------------------------------------------------

    if len(selected_indices) > summary_limit:

        ranked = sorted(
            selected_indices,
            key=lambda x: weighted_scores[x],
            reverse=True
        )

        selected_indices = ranked[:summary_limit]

    # --------------------------------------------------------
    # LEGAL DISCOURSE ORDERING
    # --------------------------------------------------------

    if config["use_legal_order"]:

        role_order = {
            role: idx
            for idx, role in enumerate(LEGAL_FLOW)
        }

        selected_indices.sort(
            key=lambda x: (
                role_order.get(roles[x], 999),
                x
            )
        )

    return selected_indices

# ============================================================
# STRUCTURED ROLE-TAGGED INPUT
# ============================================================

def build_structured_input(
    sentences,
    roles,
    selected_indices
):

    structured_input = []

    for idx in selected_indices:

        role = roles[idx]

        sentence = sentences[idx]

        structured_input.append(
            f"[{role}] {sentence}"
        )

    return " ".join(structured_input)

# ============================================================
# TEXT CHUNKING
# ============================================================

def chunk_text(
    text,
    tokenizer,
    max_tokens=900,
    stride=50
):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start = end - stride

    return chunks

# ============================================================
# CHUNK-BASED SUMMARY GENERATION
# ============================================================

def generate_summary_chunked(
    text,
    tokenizer,
    model
):

    chunks = chunk_text(
        text,
        tokenizer
    )

    chunk_summaries = []

    for chunk in chunks:

        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():

            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    return " ".join(chunk_summaries)

# ============================================================
# FACTCC EVALUATION
# ============================================================

def factcc_chunked_fast(
    prediction,
    judgement_chunks
):

    scores = []

    for chunk in judgement_chunks:

        inputs = factcc_tokenizer(
            prediction,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():

            logits = factcc_model(
                **inputs
            ).logits

            probs = F.softmax(
                logits,
                dim=-1
            )

        scores.append(
            probs[0, 1].item()
        )

    return float(np.mean(scores))

# ============================================================
# MAIN EVALUATION LOOP
# ============================================================

results = {}

for variant_name, config in ABLATION_VARIANTS.items():

    print(f"\nRunning Variant: {variant_name}")

    rouge1_scores = []
    rouge2_scores = []
    rougel_scores = []
    rougelsum_scores = []

    bleu_scores = []
    meteor_scores = []

    bert_scores = []
    bart_scores = []

    factcc_scores = []

    for doc in tqdm(dataset, desc=variant_name):

        # ----------------------------------------------------
        # REFERENCE SUMMARY
        # ----------------------------------------------------

        reference = " ".join(
            doc["headnote_sent"]
        )

        # ----------------------------------------------------
        # SOURCE JUDGMENT
        # ----------------------------------------------------

        judgement = " ".join(
            preprocess(s["sentence"])
            for s in doc["judgement_roles"]
            if isinstance(s, dict)
        )

        sentences = [
            s["sentence"]
            for s in doc["judgement_roles"]
        ]

        roles = [
            s["role"]
            for s in doc["judgement_roles"]
        ]

        # ----------------------------------------------------
        # EVIDENCE SELECTION
        # ----------------------------------------------------

        selected_indices = ralss_sentence_selection(
            sentences,
            roles,
            config
        )

        structured_input = build_structured_input(
            sentences,
            roles,
            selected_indices
        )

        # ----------------------------------------------------
        # SUMMARY GENERATION
        # ----------------------------------------------------

        prediction = generate_summary_chunked(
            structured_input,
            tokenizer,
            summarizer_model
        )

        # ----------------------------------------------------
        # ROUGE
        # ----------------------------------------------------

        rouge_result = rouge.compute(
            predictions=[prediction],
            references=[reference]
        )

        rouge1_scores.append(
            rouge_result["rouge1"]
        )

        rouge2_scores.append(
            rouge_result["rouge2"]
        )

        rougel_scores.append(
            rouge_result["rougeL"]
        )

        rougelsum_scores.append(
            rouge_result["rougeLsum"]
        )

        # ----------------------------------------------------
        # BLEU
        # ----------------------------------------------------

        bleu_result = bleu.compute(
            predictions=[prediction],
            references=[[reference]]
        )

        bleu_scores.append(
            bleu_result["bleu"]
        )

        # ----------------------------------------------------
        # METEOR
        # ----------------------------------------------------

        meteor_result = meteor.compute(
            predictions=[prediction],
            references=[reference]
        )

        meteor_scores.append(
            meteor_result["meteor"]
        )

        # ----------------------------------------------------
        # BERTScore
        # ----------------------------------------------------

        bert_result = bertscore.compute(
            predictions=[prediction],
            references=[reference],
            lang="en"
        )

        bert_scores.append(
            bert_result["f1"][0]
        )

        # ----------------------------------------------------
        # BARTScore
        # ----------------------------------------------------

        bart_scores.append(
            bart_scorer.score(
                [prediction],
                [reference]
            )[0]
        )

        # ----------------------------------------------------
        # FACTCC
        # ----------------------------------------------------

        judgement_chunks = chunk_text(
            judgement,
            tokenizer,
            max_tokens=900,
            stride=150
        )

        factcc_scores.append(
            factcc_chunked_fast(
                prediction,
                judgement_chunks
            )
        )

        # ----------------------------------------------------
        # MEMORY CLEANUP
        # ----------------------------------------------------

        torch.cuda.empty_cache()

        gc.collect()

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(f"ROUGE-1    : {np.mean(rouge1_scores):.4f}")
    print(f"ROUGE-2    : {np.mean(rouge2_scores):.4f}")
    print(f"ROUGE-L    : {np.mean(rougel_scores):.4f}")
    print(f"ROUGE-LSUM : {np.mean(rougelsum_scores):.4f}")
    print(f"BLEU       : {np.mean(bleu_scores):.4f}")
    print(f"METEOR     : {np.mean(meteor_scores):.4f}")
    print(f"BERTScore  : {np.mean(bert_scores):.4f}")
    print(f"BARTScore  : {np.mean(bart_scores):.4f}")
    print(f"FactCC     : {np.mean(factcc_scores):.4f}")

    results[variant_name] = {

        "ROUGE-1": np.mean(rouge1_scores),
        "ROUGE-2": np.mean(rouge2_scores),
        "ROUGE-L": np.mean(rougel_scores),
        "ROUGE-LSUM": np.mean(rougelsum_scores),

        "BLEU": np.mean(bleu_scores),
        "METEOR": np.mean(meteor_scores),

        "BERTScore": np.mean(bert_scores),
        "BARTScore": np.mean(bart_scores),

        "FactCC": np.mean(factcc_scores)
    }

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================================================")
print("FINAL RESULTS")
print("================================================")

for variant, metrics in results.items():

    print(f"\n{variant}")

    for metric, score in metrics.items():

        print(f"{metric}: {score:.4f}")